# 🦙 LLaMA 3.2-3B Fine-Tuning — ArXiv Abstract Generation
**Task:** Fine-tune `Llama-3.2-3B-Instruct` (4-bit) with LoRA to generate academic abstracts from paper titles.

| Setting | Value |
|---|---|
| Base model | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` |
| Method | LoRA (r=16) |
| Dataset | ArXiv parquet (nested by `primary_category`) |
| Fix | Custom `UnslothTrainer` bypassing broken `_unsloth_training_step` |

## 1. Install Dependencies

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" -q
!pip install --no-deps xformers trl peft accelerate bitsandbytes -q
print("✅ Dependencies installed.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
✅ Dependencies installed.


## 2. Imports & Configuration

In [2]:
from unsloth import FastLanguageModel
import torch
import torch.nn as nn
from datasets import load_dataset
from transformers import Trainer, TrainingArguments
from torch.nn.utils.rnn import pad_sequence
from dataclasses import dataclass
from typing import Any
import glob, os

# ── Hyperparameters ────────────────────────────────────────────────────────────
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT   = True
LORA_RANK      = 16
LORA_ALPHA     = 16
BATCH_SIZE     = 4
GRAD_ACCUM     = 2
MAX_STEPS      = 60
LEARNING_RATE  = 2e-4
WARMUP_STEPS   = 5
SEED           = 3407
OUTPUT_DIR     = "outputs"

# Dataset root — parquet files live inside primary_category=*/ subdirs
DATASET_ROOT = "/kaggle/input/datasets/rawoofraw/arxiv-dataset"

print("⚙️  Config loaded.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⚙️  Config loaded.


## 3. Discover Dataset Files

In [3]:
# Glob picks up parquet files nested under primary_category=<x>/ folders.
# The .crc sidecar files are checksums — explicitly exclude them.
data_files = [
    p for p in glob.glob(f"{DATASET_ROOT}/**/*.parquet", recursive=True)
    if not os.path.basename(p).startswith(".")   # skip hidden .crc files
]

assert len(data_files) > 0, f"No parquet files found under: {DATASET_ROOT}"

categories = set(os.path.basename(os.path.dirname(p)) for p in data_files)
print(f"📂 {len(data_files)} parquet files across {len(categories)} categories")
print(f"   Categories: {sorted(categories)[:8]} {'...' if len(categories) > 8 else ''}")
print(f"   Example path: {data_files[0]}")

📂 3603 parquet files across 152 categories
   Categories: ['primary_category=astro-ph', 'primary_category=astro-ph.CO', 'primary_category=astro-ph.EP', 'primary_category=astro-ph.GA', 'primary_category=astro-ph.HE', 'primary_category=astro-ph.IM', 'primary_category=astro-ph.SR', 'primary_category=cmp-lg'] ...
   Example path: /kaggle/input/datasets/rawoofraw/arxiv-dataset/primary_category=cs.SI/part-00028-115c4125-848d-4ecd-8436-eb63b3e8e6a4.c000.snappy.parquet


## 4. Load Base Model + LoRA

In [4]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = LOAD_IN_4BIT,
)
print(f"✅ Base model loaded | dtype: {model.dtype}")

model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = 0,
    bias           = "none",
    use_gradient_checkpointing = "unsloth",
    random_state   = SEED,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA attached | {trainable/1e6:.2f}M / {total/1e6:.0f}M params ({100*trainable/total:.2f}% trainable)")

# LLaMA has no pad token by default — use EOS
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
    print(f"ℹ️  pad_token_id set to eos_token_id ({tokenizer.eos_token_id})")

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/Llama-3.2-3B-Instruct-bnb-4bit as a legacy tokenizer.


✅ Base model loaded | dtype: torch.float16


Unsloth 2026.5.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA attached | 24.31M / 1828M params (1.33% trainable)


## 5. Prompt Template & Tokenization

In [5]:
PROMPT_TEMPLATE = """Below is a title of an academic research paper. Write a highly technical and professional abstract for this paper.
### Title:
{}
### Abstract:
{}"""

EOS_TOKEN = tokenizer.eos_token

def tokenize_and_format(examples):
    texts = [
        PROMPT_TEMPLATE.format(title, abstract) + EOS_TOKEN
        for title, abstract in zip(examples["title"], examples["clean_abstract"])
    ]
    return tokenizer(
        texts,
        truncation = True,
        max_length = MAX_SEQ_LENGTH,
        padding    = False,
    )

print("✅ Prompt template and tokenizer function ready.")

✅ Prompt template and tokenizer function ready.


## 6. Load & Tokenize Dataset

In [6]:
raw_dataset = load_dataset("parquet", data_files=data_files, split="train")
print(f"📊 Raw dataset: {len(raw_dataset):,} rows | Columns: {raw_dataset.column_names}")

dataset = raw_dataset.map(
    tokenize_and_format,
    batched        = True,
    num_proc       = 2,
    remove_columns = raw_dataset.column_names,
    desc           = "Tokenizing",
)
print(f"✅ Tokenized: {len(dataset):,} examples")
print(f"   Sample lengths: {[len(dataset[i]['input_ids']) for i in range(3)]} tokens")

Resolving data files:   0%|          | 0/3603 [00:00<?, ?it/s]

📊 Raw dataset: 492,020 rows | Columns: ['id', 'title', 'clean_abstract']
✅ Tokenized: 492,020 examples
   Sample lengths: [269, 273, 240] tokens


## 7. Tensor-Safe Collator + UnslothTrainer

### Why two fixes are needed

**Fix 1 — `CausalLMCollator`:** Guarantees every batch value is a `torch.LongTensor`.
HuggingFace's default collator can leak Python lists/ints in some version combinations.

**Fix 2 — `UnslothTrainer`:** Unsloth monkey-patches `training_step` with a version
that mishandles the loss object when using plain `Trainer` + pre-tokenized data.
Overriding `training_step` bypasses the broken patch and runs the standard
HuggingFace loop — which is fully compatible with Unsloth's LoRA model.

In [7]:
@dataclass
class CausalLMCollator:
    """Pads sequences and builds labels with -100 on padding (loss ignores pads)."""
    pad_token_id: int

    def __call__(self, features: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        input_ids      = [torch.tensor(f["input_ids"],      dtype=torch.long) for f in features]
        attention_mask = [torch.tensor(f["attention_mask"], dtype=torch.long) for f in features]

        input_ids_padded      = pad_sequence(input_ids,      batch_first=True, padding_value=self.pad_token_id)
        attention_mask_padded = pad_sequence(attention_mask, batch_first=True, padding_value=0)

        labels = input_ids_padded.clone()
        labels[attention_mask_padded == 0] = -100   # ignore padding in loss

        return {
            "input_ids":      input_ids_padded,
            "attention_mask": attention_mask_padded,
            "labels":         labels,
        }


class UnslothTrainer(Trainer):
    """
    Overrides training_step to bypass Unsloth's broken _unsloth_training_step patch.
    The patch calls .mean() on a value that is sometimes a plain Python int,
    causing: AttributeError: 'int' object has no attribute 'mean'
    This override runs the standard HuggingFace training_step instead.
    Unsloth's LoRA speed optimisations (4-bit, gradient checkpointing) are
    unaffected — only the broken training_step patch is skipped.
    """
    def training_step(
        self,
        model: nn.Module,
        inputs: dict[str, torch.Tensor],
        num_items_in_batch: int | None = None,
    ) -> torch.Tensor:
        model.train()
        inputs = self._prepare_inputs(inputs)

        with self.compute_loss_context_manager():
            loss = self.compute_loss(model, inputs)

        if self.args.n_gpu > 1:
            loss = loss.mean()

        self.accelerator.backward(loss)
        return loss.detach()


data_collator = CausalLMCollator(pad_token_id=tokenizer.pad_token_id)
print("✅ CausalLMCollator and UnslothTrainer defined.")

✅ CausalLMCollator and UnslothTrainer defined.


## 8. Initialize Trainer

In [8]:
use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
print(f"⚡ Precision: {'bfloat16' if use_bf16 else 'float16'}")

trainer = UnslothTrainer(
    model            = model,
    processing_class = tokenizer,
    train_dataset    = dataset,
    data_collator    = data_collator,
    args = TrainingArguments(
        output_dir                  = OUTPUT_DIR,
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = WARMUP_STEPS,
        max_steps                   = MAX_STEPS,
        learning_rate               = LEARNING_RATE,
        fp16                        = use_fp16,
        bf16                        = use_bf16,
        logging_steps               = 1,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        lr_scheduler_type           = "linear",
        seed                        = SEED,
        report_to                   = "none",
    ),
)
print("✅ UnslothTrainer initialized.")

⚡ Precision: float16
✅ UnslothTrainer initialized.


## 9. Train

In [9]:
print("🚀 Starting training...")
print(f"   Steps: {MAX_STEPS} | Effective batch: {BATCH_SIZE * GRAD_ACCUM} | LR: {LEARNING_RATE}")
print("-" * 60)

trainer_stats = trainer.train()

print("-" * 60)
print("✅ Training complete!")
print(f"   Steps:      {int(trainer_stats.global_step)}")
print(f"   Final loss: {trainer_stats.training_loss:.4f}")
print(f"   Time:       {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"   Peak VRAM:  {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

🚀 Starting training...
   Steps: 60 | Effective batch: 8 | LR: 0.0002
------------------------------------------------------------


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 492,020 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
Unsloth: Not an error, but LlamaForCausalLM does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,5.828450
2,6.019735
3,5.750545
4,5.828855
5,5.581818
6,5.412769
7,5.415841
8,5.655163
9,5.159545
10,5.140215


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


------------------------------------------------------------
✅ Training complete!
   Steps:      60
   Final loss: 5.0202
   Time:       469.2s
   Peak VRAM:  4.60 GB


## 10. Save LoRA Adapter

In [10]:
save_path = f"{OUTPUT_DIR}/lora_adapter"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"💾 LoRA adapter saved to '{save_path}/'")
print(f"   Files: {os.listdir(save_path)}")

# ── Optional: merge LoRA into base weights for standalone deployment ────────────
# model.save_pretrained_merged(f"{OUTPUT_DIR}/merged_model", tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in outputs/lora_adapter/tokenizer_config.json.


💾 LoRA adapter saved to 'outputs/lora_adapter/'
   Files: ['tokenizer_config.json', 'README.md', 'adapter_config.json', 'chat_template.jinja', 'adapter_model.safetensors', 'tokenizer.json']


## 11. Inference — Generate Abstracts

In [11]:
FastLanguageModel.for_inference(model)

test_titles = [
    "Generative Approaches for Biomedical Data Augmentation Using VAEs and DDPM",
    "Efficient Transformer Architectures for Long-Document Summarization",
    "Dark Matter Distribution in Elliptical Galaxies via Gravitational Lensing",
]

for title in test_titles:
    print(f"📄 Title: {title}")
    print("-" * 70)

    prompt = PROMPT_TEMPLATE.format(title, "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens = 256,
            use_cache      = True,
            temperature    = 0.7,
            do_sample      = True,
        )

    full_text = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    abstract  = full_text.split("### Abstract:")[-1].strip()
    print(f"📝 Abstract:\n{abstract}")
    print("=" * 70 + "\n")

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


📄 Title: Generative Approaches for Biomedical Data Augmentation Using VAEs and DDPM
----------------------------------------------------------------------


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 Abstract:
Biomedical data augmentation is a critical step in developing machine learning models for medical diagnosis and disease prediction. However, the lack of diverse and high-quality data for such tasks hinders the development of accurate models. In this paper, we present a novel approach to biomedical data augmentation using generative models. Specifically, we use Variational Autoencoders (VAEs) and Denoising Diffusion Processes (DDPM) for data augmentation. Our approach consists of two steps: first, we use a VAE to generate new data samples from a given dataset, and then we use DDPM to generate additional data samples from the generated data. We evaluate our approach using a case study on breast cancer diagnosis data and demonstrate its effectiveness in increasing the diversity of the training data. We also evaluate the performance of our approach on a range of machine learning models, including logistic regression, random forest, and convolutional neural networks. Our results

Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📝 Abstract:
Long document summarization is a task where we need to condense long documents into a short and meaningful summary. This task requires the model to understand the content of the document, identify the most important information, and express it in a clear and concise manner. In this paper, we present a new transformer architecture for long document summarization, which is based on the idea of self-attention. Our architecture is based on the Transformer architecture, but it is designed to handle long documents with many different lengths. We use a novel self-attention mechanism that allows the model to attend to different parts of the document simultaneously. We also introduce a novel document encoder that allows the model to encode long documents in parallel. Our architecture is efficient and effective, and it achieves state-of-the-art results on the CNN/Daily Mail and Reddit Summarization tasks.

📄 Title: Dark Matter Distribution in Elliptical Galaxies via Gravitational Len

In [12]:
# LoRA adapter only (~50MB)
model.save_pretrained("/kaggle/working/lora_adapter")
tokenizer.save_pretrained("/kaggle/working/lora_adapter")

# OR merged full model (~6GB in float16)
model.save_pretrained_merged(
    "/kaggle/working/merged_model",
    tokenizer,
    save_method = "merged_16bit"
)

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/lora_adapter/tokenizer_config.json.


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/merged_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:21<00:00, 10.67s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:42<00:00, 21.33s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/merged_model`


In [13]:
import shutil

# Zip the small adapter weights
shutil.make_archive('/kaggle/working/my_llama_academic_adapter', 'zip', '/kaggle/working/lora_adapter')
print("Adapter zipped and ready for download!")

Adapter zipped and ready for download!
